# Análise POD multivariável - caso OpenFOAM

Este notebook executa POD usando as mesmas variáveis físicas do SAE: `Ux`, `Uy`, `alpha.water`, `p_rgh` e `k`.


In [3]:
from __future__ import annotations

import os
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Tuple, List

import numpy as np

# Backend não interativo para geração reprodutível das figuras em arquivo.
os.environ["MPLBACKEND"] = "Agg"

import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
plt.switch_backend("Agg")

from scipy.interpolate import griddata


In [4]:
# =========================
# Parâmetros da análise
# =========================

DEFAULT_SRC_DIR = Path(r"/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/src")
THIS_DIR = Path(__file__).resolve().parent if "__file__" in globals() else DEFAULT_SRC_DIR
CASE_DIR = Path(r"/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000")
OUT_DIR  = THIS_DIR.parent / "resultados"

FIELD_NAMES = ("U", "alpha.water", "p_rgh", "k")
MESH_TIME_NAME = None
STRUCTURED = False

USE_ALL_TIMES = True
TIME_START = None
TIME_END   = None
TIME_STEP  = 1  # antes era 1; 10001 snapshots causavam OOM/fechamento do VS Code

RMAX: Optional[int] = 50
R_LIST: Optional[List[int]] = None
CENTER_DATA = True
NORMALIZE_SNAPSHOTS = True

GRID_NX = 220
GRID_NY = 220
GRID_METHOD = "linear"
GRID_PADDING = 0.02
MODE_PLOT_MAX = 10
RECON_TOPK = 5

USE_ROI = False
ROI_XMIN = None
ROI_XMAX = None
ROI_YMIN = None
ROI_YMAX = None

SEED = 7
TEST_FRACTION = 0.1

print(f"THIS_DIR: {THIS_DIR}")
print(f"CASE_DIR: {CASE_DIR}")
print(f"OUT_DIR : {OUT_DIR}")



THIS_DIR: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/src
CASE_DIR: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000
OUT_DIR : /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados


In [5]:
try:
    from fluidfoam import readmesh, readfield
except Exception as e:
    raise ImportError(
        "N\u00e3o foi poss\u00edvel importar fluidfoam. Instale e/ou ative o ambiente do OpenFOAM.\n"
        "pip install fluidfoam\n\nErro original:\n"
        f"{e}"
    )


In [6]:
def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def list_time_dirs(case_dir: Path) -> List[str]:
    times = []
    for item in case_dir.iterdir():
        if item.is_dir():
            try:
                float(item.name)
                times.append(item.name)
            except Exception:
                pass
    return sorted(times, key=lambda s: float(s))

def filter_times(times: List[str]) -> List[str]:
    out = times
    if TIME_START is not None:
        out = [t for t in out if float(t) >= float(TIME_START)]
    if TIME_END is not None:
        out = [t for t in out if float(t) <= float(TIME_END)]
    return out[:: max(1, int(TIME_STEP))]

def read_mesh_xy(case_dir: Path) -> Tuple[np.ndarray, np.ndarray]:
    mesh = readmesh(str(case_dir), time_name=MESH_TIME_NAME, structured=STRUCTURED)
    if isinstance(mesh, (tuple, list)) and len(mesh) >= 2:
        return np.asarray(mesh[0]).ravel(), np.asarray(mesh[1]).ravel()
    mesh = np.asarray(mesh)
    if mesh.ndim == 2 and mesh.shape[1] >= 2:
        return mesh[:, 0].ravel(), mesh[:, 1].ravel()
    raise ValueError("Não foi possível interpretar a malha com fluidfoam.readmesh().")

def read_U_components(case_dir: Path, time_name: str, nCells: int) -> Tuple[np.ndarray, np.ndarray]:
    Uf = np.asarray(readfield(str(case_dir), time_name, "U"))
    if Uf.ndim == 2 and Uf.shape == (3, 1):
        return np.full(nCells, float(Uf[0, 0])), np.full(nCells, float(Uf[1, 0]))
    if Uf.ndim == 2 and Uf.shape[0] == 3 and Uf.shape[1] == nCells:
        return Uf[0, :].astype(float), Uf[1, :].astype(float)
    if Uf.ndim == 2 and Uf.shape[0] == nCells and Uf.shape[1] >= 2:
        return Uf[:, 0].astype(float), Uf[:, 1].astype(float)
    if Uf.ndim == 1 and Uf.size == 3 * nCells:
        Uf = Uf.reshape(nCells, 3)
        return Uf[:, 0].astype(float), Uf[:, 1].astype(float)
    raise ValueError(f"Formato inesperado de U em {time_name}: shape={Uf.shape}, nCells={nCells}")

def read_scalar_cell_field(case_dir: Path, time_name: str, field_name: str, nCells: int) -> np.ndarray:
    try:
        field = np.asarray(readfield(str(case_dir), time_name, field_name), dtype=float)
    except Exception:
        return np.zeros(nCells, dtype=float)
    field = np.squeeze(field)
    if field.ndim == 0:
        return np.full(nCells, float(field), dtype=float)
    if field.ndim == 1 and field.size == nCells:
        return field.astype(float)
    if field.ndim == 2 and 1 in field.shape and field.size == nCells:
        return field.reshape(nCells).astype(float)
    if field.size == 1:
        return np.full(nCells, float(field.ravel()[0]), dtype=float)
    raise ValueError(f"Formato inesperado de {field_name} em {time_name}: shape={field.shape}, nCells={nCells}")

def read_cell_channels(case_dir: Path, time_name: str, field_names: Tuple[str, ...], nCells: int) -> Tuple[List[np.ndarray], List[str]]:
    channels: List[np.ndarray] = []
    names: List[str] = []
    for field in field_names:
        if field == "U":
            ux, uy = read_U_components(case_dir, time_name, nCells)
            channels.extend([ux, uy])
            names.extend(["Ux", "Uy"])
        else:
            channels.append(read_scalar_cell_field(case_dir, time_name, field, nCells))
            names.append(field)
    return channels, names

def make_grid(x: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray, Tuple[float,float,float,float]]:
    xmin, xmax = float(np.min(x)), float(np.max(x))
    ymin, ymax = float(np.min(y)), float(np.max(y))
    dx = (xmax - xmin) * GRID_PADDING
    dy = (ymax - ymin) * GRID_PADDING
    gx = np.linspace(xmin - dx, xmax + dx, GRID_NX)
    gy = np.linspace(ymin - dy, ymax + dy, GRID_NY)
    Xg, Yg = np.meshgrid(gx, gy)
    return Xg, Yg, (xmin - dx, xmax + dx, ymin - dy, ymax + dy)

def interp_to_grid(x: np.ndarray, y: np.ndarray, val: np.ndarray, Xg: np.ndarray, Yg: np.ndarray) -> np.ndarray:
    pts = np.column_stack([x, y])
    Vg = griddata(pts, val, (Xg, Yg), method=GRID_METHOD)
    if np.isnan(Vg).any():
        Vn = griddata(pts, val, (Xg, Yg), method="nearest")
        Vg = np.where(np.isnan(Vg), Vn, Vg)
    return Vg

def apply_roi_mask(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    mask = np.ones_like(x, dtype=bool)
    if ROI_XMIN is not None: mask &= (x >= ROI_XMIN)
    if ROI_XMAX is not None: mask &= (x <= ROI_XMAX)
    if ROI_YMIN is not None: mask &= (y >= ROI_YMIN)
    if ROI_YMAX is not None: mask &= (y <= ROI_YMAX)
    return mask

def safe_rel(a: np.ndarray, b: np.ndarray, eps: float = 1e-12) -> float:
    return float(np.linalg.norm(a - b) / (np.linalg.norm(a) + eps))

def compute_pod_svd(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    return np.linalg.svd(X, full_matrices=False)

def reconstruct_from_svd(U: np.ndarray, s: np.ndarray, Vt: np.ndarray, r: int) -> np.ndarray:
    return (U[:, :r] * s[:r]) @ Vt[:r, :]

def channel_slice(channel_index: int, nCells: int) -> slice:
    return slice(channel_index * nCells, (channel_index + 1) * nCells)

def finite_minmax(*arrs: np.ndarray) -> Tuple[float, float]:
    vals = np.concatenate([np.asarray(a)[np.isfinite(a)].ravel() for a in arrs])
    if vals.size == 0:
        return 0.0, 1.0
    vmin, vmax = float(vals.min()), float(vals.max())
    if np.isclose(vmin, vmax):
        delta = max(abs(vmin), 1.0) * 1e-6
        vmin -= delta
        vmax += delta
    return vmin, vmax

def velocity_magnitude_from_vector(vec: np.ndarray, channel_names: List[str], nCells: int) -> Optional[np.ndarray]:
    if "Ux" not in channel_names or "Uy" not in channel_names:
        return None
    ix, iy = channel_names.index("Ux"), channel_names.index("Uy")
    ux = vec[channel_slice(ix, nCells)]
    uy = vec[channel_slice(iy, nCells)]
    return np.sqrt(ux**2 + uy**2)

def analyze_errors(X: np.ndarray, Xmean: np.ndarray, U: np.ndarray, s: np.ndarray, Vt: np.ndarray,
                   times: List[str], x: np.ndarray, y: np.ndarray, out_dir: Path,
                   r_values: List[int]) -> Dict[str, np.ndarray]:
    Nt = X.shape[1]
    err_tr = np.zeros((Nt, len(r_values)), dtype=float)
    err_r = np.zeros(len(r_values), dtype=float)
    energy_cum = np.cumsum(s**2) / (np.sum(s**2) + 1e-18)
    for j, r in enumerate(r_values):
        Xr = reconstruct_from_svd(U, s, Vt, r)
        for t in range(Nt):
            err_tr[t, j] = safe_rel(X[:, t], Xr[:, t])
        err_r[j] = np.linalg.norm(X - Xr) / (np.linalg.norm(X) + 1e-18)
    plt.figure(); plt.plot(r_values, err_r, marker="o"); plt.xlabel("Número de modos POD (r)"); plt.ylabel("Erro relativo global"); plt.title("Erro global de reconstrução POD multivariável"); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig(out_dir / "erro_global_reconstrucao_vs_numero_modos.png", dpi=220); plt.close()
    plt.figure(); plt.imshow(err_tr, aspect="auto", origin="lower"); plt.colorbar(label="Erro relativo por instante"); plt.yticks(range(0, Nt, max(1, Nt // 10)), [times[i] for i in range(0, Nt, max(1, Nt // 10))]); plt.xticks(range(len(r_values)), [str(r) for r in r_values], rotation=45); plt.xlabel("Número de modos POD (r)"); plt.ylabel("Instante"); plt.title("Erro temporal POD multivariável"); plt.tight_layout(); plt.savefig(out_dir / "mapa_erro_temporal_vs_numero_modos.png", dpi=220); plt.close()
    energy_at_r = np.array([energy_cum[r-1] if r-1 < len(energy_cum) else 1.0 for r in r_values], dtype=float)
    plt.figure(); plt.plot(r_values, energy_at_r, marker="o"); plt.xlabel("Número de modos POD (r)"); plt.ylabel("Energia acumulada"); plt.title("Energia acumulada POD"); plt.ylim(0, 1.02); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig(out_dir / "energia_acumulada_vs_numero_modos.png", dpi=220); plt.close()
    return {"err_tr": err_tr, "err_r": err_r, "energy_cum": energy_cum, "energy_at_r": energy_at_r}

def save_mode_heatmaps(U_modes: np.ndarray, x: np.ndarray, y: np.ndarray, out_modes: Path,
                       nCells: int, Xg: np.ndarray, Yg: np.ndarray, n_modes_save: int,
                       channel_names: List[str]) -> None:
    ensure_dir(out_modes)
    nsave = min(n_modes_save, U_modes.shape[1])
    n_channels = len(channel_names)
    for k in range(nsave):
        phi = U_modes[:, k]
        panels = []
        for ci, name in enumerate(channel_names):
            panels.append((name, phi[channel_slice(ci, nCells)]))
        umag = velocity_magnitude_from_vector(phi, channel_names, nCells)
        if umag is not None:
            panels.append(("|U|", umag))
        n_cols = min(3, len(panels))
        n_rows = int(np.ceil(len(panels) / n_cols))
        fig, axs = plt.subplots(n_rows, n_cols, figsize=(4.6*n_cols, 3.8*n_rows), squeeze=False)
        for idx, (name, values) in enumerate(panels):
            ax = axs[idx // n_cols, idx % n_cols]
            grid = interp_to_grid(x, y, values, Xg, Yg)
            im = ax.imshow(grid, origin="lower", aspect="auto")
            ax.set_title(f"Modo {k+1} - {name}")
            ax.set_xticks([]); ax.set_yticks([])
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        for idx in range(len(panels), n_rows*n_cols):
            axs[idx // n_cols, idx % n_cols].axis("off")
        fig.tight_layout(); fig.savefig(out_modes / f"mode_{k+1:03d}_channels.png", dpi=220); plt.close(fig)

def save_reconstruction_triplets(Xtrue: np.ndarray, Xrecon: np.ndarray, Xmean: np.ndarray,
                                 times: List[str], x: np.ndarray, y: np.ndarray,
                                 out_recon: Path, nCells: int, Xg: np.ndarray, Yg: np.ndarray,
                                 topk: int, snapshot_norms: Optional[np.ndarray] = None,
                                 channel_names: Optional[List[str]] = None) -> None:
    ensure_dir(out_recon)
    channel_names = channel_names or ["Ux", "Uy"]
    Nt = Xtrue.shape[1]
    scale = np.ones(Nt, dtype=float) if snapshot_norms is None else np.asarray(snapshot_norms, dtype=float).ravel()
    if scale.size != Nt:
        raise ValueError(f"snapshot_norms.size={scale.size} diferente de Nt={Nt}")
    Xtrue_phys = (Xtrue + Xmean[:, None]) * scale[None, :]
    Xrecon_phys = (Xrecon + Xmean[:, None]) * scale[None, :]
    err_t = np.array([safe_rel(Xtrue_phys[:, t], Xrecon_phys[:, t]) for t in range(Nt)], dtype=float)
    worst = np.argsort(-err_t)[: min(topk, Nt)]
    for rank_i, t in enumerate(worst, start=1):
        Xt, Xr = Xtrue_phys[:, t], Xrecon_phys[:, t]
        panels = [(name, Xt[channel_slice(ci, nCells)], Xr[channel_slice(ci, nCells)]) for ci, name in enumerate(channel_names)]
        umag_t = velocity_magnitude_from_vector(Xt, channel_names, nCells)
        umag_r = velocity_magnitude_from_vector(Xr, channel_names, nCells)
        if umag_t is not None and umag_r is not None:
            panels.append(("|U|", umag_t, umag_r))
        for name, a, b in panels:
            e = b - a
            A, B, E = interp_to_grid(x, y, a, Xg, Yg), interp_to_grid(x, y, b, Xg, Yg), interp_to_grid(x, y, e, Xg, Yg)
            vmin, vmax = finite_minmax(A, B)
            emax = float(np.nanmax(np.abs(E))) if np.isfinite(E).any() else 1.0
            if emax <= 0 or not np.isfinite(emax): emax = 1.0
            fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
            im0 = axs[0].imshow(A, origin="lower", aspect="auto", vmin=vmin, vmax=vmax, cmap="viridis"); axs[0].set_title(f"{name} original")
            im1 = axs[1].imshow(B, origin="lower", aspect="auto", vmin=vmin, vmax=vmax, cmap="viridis"); axs[1].set_title(f"{name} POD")
            im2 = axs[2].imshow(E, origin="lower", aspect="auto", vmin=-emax, vmax=emax, cmap="coolwarm"); axs[2].set_title(f"erro {name}")
            for im, ax in [(im0, axs[0]), (im1, axs[1]), (im2, axs[2])]:
                ax.set_xticks([]); ax.set_yticks([]); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            fig.suptitle(f"POD multivariável | t={times[t]} s | L2={err_t[t]:.3e}")
            fig.tight_layout(); fig.savefig(out_recon / f"worst_{rank_i:02d}_t{times[t].replace('.','p')}_{name.replace('|','').replace('/','_')}.png", dpi=220); plt.close(fig)


In [7]:
# Etapa 1: preparação dos diretórios de saída, instantes de tempo e malha
ensure_dir(OUT_DIR)
out_modes = OUT_DIR / "figuras_modos_pod"
out_recon = OUT_DIR / "figuras_reconstrucao_maiores_erros"
ensure_dir(out_modes)
ensure_dir(out_recon)

times_all = list_time_dirs(CASE_DIR)
times = filter_times(times_all)
if not times:
    raise RuntimeError(f"Nenhum diret\u00f3rio de tempo encontrado/selecionado em {CASE_DIR}")

x, y = read_mesh_xy(CASE_DIR)
if x.ndim != 1 or y.ndim != 1:
    raise ValueError(f"Malha inv\u00e1lida: x.ndim={x.ndim}, y.ndim={y.ndim}. Esperado vetores 1D.")
if x.size != y.size:
    raise ValueError(f"Malha inconsistente: x.size={x.size} != y.size={y.size}")

nCells = x.size
if nCells <= 0:
    raise ValueError("N\u00famero de c\u00e9lulas inv\u00e1lido: nCells <= 0")

Xg, Yg, _bbox = make_grid(x, y)
Nt = len(times)
first_channels, CHANNEL_NAMES = read_cell_channels(CASE_DIR, times[0], FIELD_NAMES, nCells)
nChannels = len(CHANNEL_NAMES)
Xsnap = np.zeros((nChannels * nCells, Nt), dtype=float)

print("[Verificação] Malha")
print(f"x.shape = {x.shape}")
print(f"y.shape = {y.shape}")
print(f"nCells  = {nCells}")
print("[Verificação] Grade de interpolação")
print(f"Xg.shape = {Xg.shape}")
print(f"Yg.shape = {Yg.shape}")
print("[Verificação] Matriz de snapshots")
print(f"Nt = {Nt}")
print(f"CHANNEL_NAMES = {CHANNEL_NAMES}")
print(f"nChannels = {nChannels}")
print(f"Xsnap.shape = {Xsnap.shape}")


Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/constant/polyMesh/owner
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/constant/polyMesh/faces
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/constant/polyMesh/points
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/constant/polyMesh/neighbour
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0/U

Only constant field in output

Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0/alpha.water
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0/p_rgh

Only constant field in output

Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0/k
[Verificação] Malha
x.shape = (2268,)
y.shape = (2268,)
nCells  = 2268
[Verificação] Grade de interpo

In [8]:
# Etapa 2: carregamento dos snapshots multivariáveis e pré-processamento
for i, tname in enumerate(times):
    channels, names = read_cell_channels(CASE_DIR, tname, FIELD_NAMES, nCells)
    if names != CHANNEL_NAMES:
        raise ValueError(f"Canais inconsistentes em {tname}: {names} != {CHANNEL_NAMES}")
    for ci, values in enumerate(channels):
        values = np.asarray(values, dtype=float).ravel()
        if values.size != nCells:
            raise ValueError(f"Canal {CHANNEL_NAMES[ci]} em {tname}: size={values.size}, nCells={nCells}")
        Xsnap[channel_slice(ci, nCells), i] = values

print(f"Xsnap.shape (preenchido) = {Xsnap.shape}")
print(f"Variáveis POD: {CHANNEL_NAMES}")

if NORMALIZE_SNAPSHOTS:
    norms = np.linalg.norm(Xsnap, axis=0) + 1e-18
    Xsnap = Xsnap / norms
    print("NORMALIZE_SNAPSHOTS = True")
    print(f"norms.shape = {norms.shape}")
else:
    norms = np.ones(Nt, dtype=float)
    print("NORMALIZE_SNAPSHOTS = False")

Xmean = np.zeros((nChannels * nCells,), dtype=float)
X = Xsnap.copy()

if CENTER_DATA:
    Xmean = np.mean(Xsnap, axis=1)
    X = Xsnap - Xmean[:, None]

print(f"CENTER_DATA = {CENTER_DATA}")
print(f"Xmean.shape = {Xmean.shape}")
print(f"X.shape     = {X.shape}")


Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0/U

Only constant field in output

Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0/alpha.water
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0/p_rgh

Only constant field in output

Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0/k
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0.0001/U
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0.0001/alpha.water
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0.0001/p_rgh
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0.0001/k
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/0.0002/U
Reading file /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubun

In [9]:
# Etapa 3: decomposição POD/SVD e geração dos resultados principais
U, s, Vt = compute_pod_svd(X)
rank = len(s)

print("[Verificação] Decomposição SVD")
print(f"U.shape  = {U.shape}")
print(f"s.shape  = {s.shape}")
print(f"Vt.shape = {Vt.shape}")
print(f"rank     = {rank}")

if R_LIST is not None:
    r_values = sorted({int(r) for r in R_LIST if int(r) >= 1})
    if not r_values:
        raise ValueError("R_LIST foi definido, mas n\u00e3o possui valores v\u00e1lidos >= 1")
    if max(r_values) > rank:
        raise ValueError(f"R_LIST cont\u00e9m valor maior que o rank: max(R_LIST)={max(r_values)} > rank={rank}")
    rmax_use = max(r_values)
else:
    rmax_use = rank if RMAX is None else min(int(RMAX), rank)
    r_values = list(range(1, rmax_use + 1))

print(f"rmax_use = {rmax_use}")
print(f"r_final  = {r_values[-1]}")

energy = (s**2) / (np.sum(s**2) + 1e-18)
cum_energy = np.cumsum(energy)

plt.figure()
n_energy = min(rmax_use, len(cum_energy))
plt.plot(range(1, n_energy + 1), cum_energy[:n_energy], marker="o")
plt.xlabel("Modo")
plt.ylabel("Energia acumulada")
plt.title("Energia acumulada dos modos POD")
plt.ylim(0, 1.02)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "energia_acumulada_modos_pod.png", dpi=220)
plt.close()

A = (s[:, None] * Vt)

nm = min(10, A.shape[0])
plt.figure(figsize=(10, 5))
for k in range(nm):
    plt.plot(A[k, :], label=f"modo {k+1}")
plt.xlabel("Índice do instante de tempo")
plt.ylabel("Coeficiente")
plt.title("Coeficientes temporais dos primeiros modos POD")
plt.grid(True, alpha=0.3)
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "coeficientes_temporais_modos_pod.png", dpi=220)
plt.close()

np.savez(
    OUT_DIR / "resultados_pod_basicos.npz",
    U=U, s=s, Vt=Vt,
    times=np.array(times),
    x=x, y=y,
    Xmean=Xmean,
    channel_names=np.array(CHANNEL_NAMES, dtype=object),
    field_names=np.array(FIELD_NAMES, dtype=object),
    nChannels=np.array([nChannels], dtype=int),
    centered=np.array([CENTER_DATA], dtype=bool),
)

print(f"Resultados parciais salvos em: {OUT_DIR}")


[Verificação] Decomposição SVD
U.shape  = (11340, 10001)
s.shape  = (10001,)
Vt.shape = (10001, 10001)
rank     = 10001
rmax_use = 50
r_final  = 50


/home/mlep-pc-ubuntu/.local/lib/python3.12/site-packages/IPython/core/pylabtools.py:77: DeprecationWarning: backend2gui is deprecated since IPython 8.24, backends are managed in matplotlib and can be externally registered.
  warnings.warn(
/home/mlep-pc-ubuntu/.local/lib/python3.12/site-packages/IPython/core/pylabtools.py:77: DeprecationWarning: backend2gui is deprecated since IPython 8.24, backends are managed in matplotlib and can be externally registered.
  warnings.warn(


Resultados parciais salvos em: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados


In [10]:
# Etapa 4: análise de erro, mapas dos modos e reconstrução POD
metrics = analyze_errors(X, Xmean, U, s, Vt, times, x, y, OUT_DIR, r_values)

save_mode_heatmaps(
    U_modes=U[:, :rmax_use],
    x=x, y=y,
    out_modes=out_modes,
    nCells=nCells,
    Xg=Xg, Yg=Yg,
    n_modes_save=min(MODE_PLOT_MAX, rmax_use),
    channel_names=CHANNEL_NAMES,
)

r_final = r_values[-1]
Xrecon = reconstruct_from_svd(U, s, Vt, r_final)

err_global_rfinal = np.linalg.norm(X - Xrecon) / (np.linalg.norm(X) + 1e-18)
err_t_rfinal = np.array([safe_rel(X[:, t], Xrecon[:, t]) for t in range(Nt)], dtype=float)

snapshot_norms_for_plots = norms if bool(NORMALIZE_SNAPSHOTS) else np.ones(Nt, dtype=float)

save_reconstruction_triplets(
    Xtrue=X,
    Xrecon=Xrecon,
    Xmean=Xmean,
    times=times,
    x=x, y=y,
    out_recon=out_recon,
    nCells=nCells,
    Xg=Xg, Yg=Yg,
    topk=RECON_TOPK,
    snapshot_norms=snapshot_norms_for_plots,
    channel_names=CHANNEL_NAMES,
)

np.savez(
    OUT_DIR / "resultados_pod_completos.npz",
    times=np.array(times),
    x=x, y=y,
    Xsnap=Xsnap,
    Xcentered=X,
    Xmean=Xmean,
    centered=np.array([CENTER_DATA], dtype=bool),
    U=U, s=s, Vt=Vt,
    A=A,
    Xrecon=Xrecon,
    err_global_rfinal=np.array([err_global_rfinal], dtype=float),
    err_t_rfinal_check=err_t_rfinal,
    nCells=np.array([nCells], dtype=int),
    nChannels=np.array([nChannels], dtype=int),
    channel_names=np.array(CHANNEL_NAMES, dtype=object),
    field_names=np.array(FIELD_NAMES, dtype=object),
    Nt=np.array([Nt], dtype=int),
    rank=np.array([rank], dtype=int),
    r_final=np.array([r_final], dtype=int),
    **metrics,
)

print("Análise POD finalizada com sucesso.")
print(f"Resultados salvos em: {OUT_DIR}")
print("- Curvas de energia e erro: arquivos PNG descritivos no diretório de resultados")
print(f"- Mapas dos modos POD: {out_modes}/mode_###_triplet.png")
print(f"- Reconstruções com maiores erros relativos em escala física: {out_recon}/worst_##_t*_*.png")
print(f"- Arquivo NPZ completo: {OUT_DIR / 'resultados_pod_completos.npz'}")


Análise POD finalizada com sucesso.
Resultados salvos em: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados
- Curvas de energia e erro: arquivos PNG descritivos no diretório de resultados
- Mapas dos modos POD: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados/figuras_modos_pod/mode_###_triplet.png
- Reconstruções com maiores erros relativos em escala física: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados/figuras_reconstrucao_maiores_erros/worst_##_t*_*.png
- Arquivo NPZ completo: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados/resultados_pod_completos.npz


In [11]:
# Figuras comparativas da reconstrução POD multivariável em 5 instantes selecionados
# Saídas geradas:
#   1) imagens mescladas, com todas as variáveis juntas, em out_dir
#   2) pastas por variável/caso, com imagens separadas: U, Ux, Uy, alpha_water, p_rgh, k etc.
# Correção: quando o tempo exato não existe nos snapshots, usa o snapshot mais próximo.

targets = np.array([0.00, 0.25, 0.50, 0.75, 1.00], dtype=float)
tol = 1e-6
required = ["OUT_DIR", "times", "Xsnap", "Xmean", "nCells", "x", "y", "CHANNEL_NAMES", "CENTER_DATA", "NORMALIZE_SNAPSHOTS"]
missing_req = [k for k in required if k not in globals()]
if missing_req:
    raise RuntimeError(f"Variáveis POD ausentes: {missing_req}")

if "Xrecon" not in globals():
    rr = int(r_final) if "r_final" in globals() else int(r_values[-1])
    Xrecon = reconstruct_from_svd(U, s, Vt, rr)
if "Xg" not in globals() or "Yg" not in globals():
    Xg, Yg, _bbox = make_grid(x, y)

times_float = np.array([float(str(tt).strip()) for tt in times], dtype=float)
norms_vec = np.asarray(norms if bool(NORMALIZE_SNAPSHOTS) else np.ones(Xsnap.shape[1]), dtype=float).ravel()

if len(times_float) == 0:
    raise RuntimeError("Nenhum snapshot de tempo disponível em 'times'.")

run_tag = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
out_dir = Path(OUT_DIR) / f"figuras_comparativas_pod_multivariavel_tempos_0a1_{run_tag}"
out_dir.mkdir(parents=True, exist_ok=False)
phys_out = out_dir  # usado pelo bloco seguinte do painel temporal
print(f"Diretório das figuras comparativas: {out_dir}")

def folder_safe_name(name: str) -> str:
    """Nome seguro para pasta/arquivo."""
    return (
        str(name)
        .replace("|U|", "U")
        .replace("alpha.water", "alpha_water")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("|", "")
    )

def comparison_panels(vec_true: np.ndarray, vec_recon: np.ndarray) -> List[Tuple[str, np.ndarray, np.ndarray]]:
    panels = []
    for ci, name in enumerate(CHANNEL_NAMES):
        panels.append((name, vec_true[channel_slice(ci, nCells)], vec_recon[channel_slice(ci, nCells)]))

    # Também gera a velocidade resultante |U| na pasta U.
    umag_t = velocity_magnitude_from_vector(vec_true, CHANNEL_NAMES, nCells)
    umag_r = velocity_magnitude_from_vector(vec_recon, CHANNEL_NAMES, nCells)
    if umag_t is not None and umag_r is not None:
        panels.append(("|U|", umag_t, umag_r))
    return panels

def save_one_comparison_panel(
    name: str,
    true_values: np.ndarray,
    recon_values: np.ndarray,
    outpath: Path,
    target_time: float,
    snapshot_time: float,
    rel_l2_var: float,
) -> None:
    """Salva uma figura separada de uma variável: OpenFOAM, POD e erro."""
    outpath.parent.mkdir(parents=True, exist_ok=True)

    err_values = recon_values - true_values
    A = interp_to_grid(x, y, true_values, Xg, Yg)
    B = interp_to_grid(x, y, recon_values, Xg, Yg)
    E = interp_to_grid(x, y, err_values, Xg, Yg)

    vmin, vmax = finite_minmax(A, B)
    emax = float(np.nanmax(np.abs(E))) if np.isfinite(E).any() else 1.0
    if emax <= 0 or not np.isfinite(emax):
        emax = 1.0

    fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.4), squeeze=False)
    title_suffix = f"alvo={target_time:.2f}s | snap={snapshot_time:.6g}s"
    specs = (
        (A, f"{name} OpenFOAM\n{title_suffix}", "viridis", vmin, vmax),
        (B, f"{name} POD\n{title_suffix}", "viridis", vmin, vmax),
        (E, f"{name} erro\nL2={rel_l2_var:.4e}", "coolwarm", -emax, emax),
    )

    for col, (img, title, cmap, cmin, cmax) in enumerate(specs):
        im = ax[0, col].imshow(img, origin="lower", cmap=cmap, vmin=cmin, vmax=cmax, interpolation="bicubic")
        ax[0, col].set_title(title, fontsize=10)
        ax[0, col].axis("off")
        fig.colorbar(im, ax=ax[0, col], fraction=0.046, pad=0.04)

    fig.suptitle(f"Reconstrução POD - {name}", fontsize=13)
    fig.tight_layout()
    fig.savefig(outpath, dpi=220)
    plt.close(fig)

def save_img_pod_multivar(
    Xt_phys: np.ndarray,
    Xr_phys: np.ndarray,
    outpath: Path,
    target_time: float,
    snapshot_time: float,
    rel_l2: float,
) -> None:
    """Salva a imagem mesclada com todas as variáveis juntas."""
    panels = comparison_panels(Xt_phys, Xr_phys)
    fig, ax = plt.subplots(len(panels), 3, figsize=(12, 3.4 * len(panels)), squeeze=False)
    title_suffix = f"alvo={target_time:.2f}s | snap={snapshot_time:.6g}s"

    for row, (name, a, b) in enumerate(panels):
        e = b - a
        A = interp_to_grid(x, y, a, Xg, Yg)
        B = interp_to_grid(x, y, b, Xg, Yg)
        E = interp_to_grid(x, y, e, Xg, Yg)
        vmin, vmax = finite_minmax(A, B)
        emax = float(np.nanmax(np.abs(E))) if np.isfinite(E).any() else 1.0
        if emax <= 0 or not np.isfinite(emax):
            emax = 1.0

        specs = (
            (A, f"{name} OpenFOAM {title_suffix}", "viridis", vmin, vmax),
            (B, f"{name} POD {title_suffix}", "viridis", vmin, vmax),
            (E, f"{name} erro {title_suffix}", "coolwarm", -emax, emax),
        )
        for col, (img, title, cmap, cmin, cmax) in enumerate(specs):
            im = ax[row, col].imshow(img, origin="lower", cmap=cmap, vmin=cmin, vmax=cmax, interpolation="bicubic")
            ax[row, col].set_title(title, fontsize=9)
            ax[row, col].axis("off")
            fig.colorbar(im, ax=ax[row, col], fraction=0.046, pad=0.04)

    fig.suptitle(f"Reconstrução POD multivariável - L2 relativo = {rel_l2:.4e}", fontsize=12)
    fig.tight_layout()
    fig.savefig(outpath, dpi=220)
    plt.close(fig)

def nearest_snapshot_index(target_time: float) -> int:
    exact = np.flatnonzero(np.isclose(times_float, target_time, atol=tol, rtol=0.0))
    if exact.size > 0:
        return int(exact[0])
    return int(np.argmin(np.abs(times_float - target_time)))

print("=== Figuras comparativas POD multivariável ===")
print("Será gerada uma versão mesclada e também pastas separadas por variável/caso.")

for tt in targets:
    i = nearest_snapshot_index(float(tt))
    snapshot_time = float(times_float[i])

    if not np.isclose(snapshot_time, tt, atol=tol, rtol=0.0):
        print(f"[Aviso] t={tt:.2f} não existe. Usando snapshot mais próximo: t={snapshot_time:.6g}")

    Xt_snap = np.asarray(Xsnap[:, i], dtype=float)
    Xr_snap = np.asarray(Xrecon[:, i], dtype=float)
    if bool(CENTER_DATA):
        Xr_snap = Xr_snap + np.asarray(Xmean, dtype=float)

    scale_i = float(norms_vec[i])
    Xt_phys = Xt_snap * scale_i
    Xr_phys = Xr_snap * scale_i
    rel_p = safe_rel(Xt_phys, Xr_phys)

    label_tt = f"{tt:.2f}".replace(".", "p")

    # 1) Imagem mesclada com todas as variáveis juntas.
    merged_outpath = out_dir / f"comparacao_pod_multivariavel_t_{label_tt}.png"
    save_img_pod_multivar(Xt_phys, Xr_phys, merged_outpath, float(tt), snapshot_time, rel_p)
    print(f"Figura mesclada salva: {merged_outpath} | alvo={tt:.2f} | snapshot={snapshot_time:.6g} | L2={rel_p:.4e}")

    # 2) Imagens separadas por variável/caso, cada uma em sua pasta.
    for var_name, true_values, recon_values in comparison_panels(Xt_phys, Xr_phys):
        var_folder = out_dir / folder_safe_name(var_name)
        var_rel = safe_rel(true_values, recon_values)
        var_file = var_folder / f"comparacao_{folder_safe_name(var_name)}_t_{label_tt}.png"
        save_one_comparison_panel(var_name, true_values, recon_values, var_file, float(tt), snapshot_time, var_rel)

# Verificação final da versão mesclada: o painel temporal encontrará exatamente 5 figuras.
generated = sorted(out_dir.glob("comparacao_pod_multivariavel_t_*.png"))
if len(generated) != 5:
    raise RuntimeError(f"Erro interno: deveriam existir 5 figuras mescladas, mas foram geradas {len(generated)} em {out_dir}")

print("\nPastas por variável/caso geradas em:")
for d in sorted([p for p in out_dir.iterdir() if p.is_dir()]):
    print(f"  {d}")



Diretório das figuras comparativas: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados/figuras_comparativas_pod_multivariavel_tempos_0a1_20260928_064424_008286
=== Figuras comparativas POD multivariável ===
Será gerada uma versão mesclada e também pastas separadas por variável/caso.
Figura mesclada salva: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados/figuras_comparativas_pod_multivariavel_tempos_0a1_20260928_064424_008286/comparacao_pod_multivariavel_t_0p00.png | alvo=0.00 | snapshot=0 | L2=6.5918e-07
Figura mesclada salva: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados/figuras_comparativas_pod_multivariavel_tempos_0a1_20260928_064424_008286/comparacao_pod_multivariavel_t_0p25.png | alvo=0.25 | snapshot=0.25 | L2=4.3464e-02
Figura mesclada salva: /home/mlep-pc-ubuntu/Ope

In [12]:
# ============================================================
# Painel temporal das figuras comparativas da reconstrução POD
# Organização: três imagens na primeira linha e duas na segunda linha
# ============================================================
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import FancyBboxPatch

# Seleciona automaticamente o diretório mais recente com as figuras comparativas.
if "phys_out" in globals():
    img_dir = Path(phys_out)
elif "OUT_DIR" in globals():
    candidates = sorted(Path(OUT_DIR).glob("figuras_comparativas_pod_multivariavel_tempos_0a1_*"))
    img_dir = candidates[-1] if candidates else Path(OUT_DIR)
else:
    img_dir = Path.cwd()

out_path = img_dir / "painel_temporal_reconstrucao_pod.png"

if not img_dir.exists():
    raise FileNotFoundError(f"Pasta nao encontrada: {img_dir}")

img_paths = sorted(img_dir.glob("comparacao_pod_multivariavel_t_*.png"))
if not img_paths:
    raise RuntimeError(f"Nenhuma figura comparativa encontrada em: {img_dir}")

def extract_time(path: Path) -> float:
    m = re.search(r"comparacao_pod_multivariavel_t_(\d+)p(\d+)\.png$", path.name)
    if m is None:
        return np.inf
    return float(f"{m.group(1)}.{m.group(2)}")

def crop_white_border_soft(
    img: np.ndarray,
    white_thr: float = 0.992,
    pad_top: int = 18,
    pad_bottom: int = 10,
    pad_left: int = 10,
    pad_right: int = 10,
) -> np.ndarray:
    arr = np.asarray(img)

    if arr.dtype.kind in "ui":
        arr = arr.astype(np.float32) / 255.0
    else:
        arr = arr.astype(np.float32)

    if arr.ndim == 2:
        gray = arr
        alpha = np.ones_like(gray)
    elif arr.shape[2] == 4:
        gray = arr[..., :3].mean(axis=2)
        alpha = arr[..., 3]
    else:
        gray = arr[..., :3].mean(axis=2)
        alpha = np.ones_like(gray)

    mask = (gray < white_thr) & (alpha > 1e-6)

    if not np.any(mask):
        return img

    ys, xs = np.where(mask)
    y0 = max(int(ys.min()) - pad_top, 0)
    y1 = min(int(ys.max()) + pad_bottom + 1, arr.shape[0])
    x0 = max(int(xs.min()) - pad_left, 0)
    x1 = min(int(xs.max()) + pad_right + 1, arr.shape[1])

    return img[y0:y1, x0:x1]

img_paths = sorted(img_paths, key=extract_time)
times = [extract_time(p) for p in img_paths]

if len(img_paths) != 5:
    raise RuntimeError(
        f"Este bloco requer 5 figuras comparativas. Encontradas: {len(img_paths)} em {img_dir}"
    )

print("Figuras encontradas em ordem temporal:")
for p, tt in zip(img_paths, times):
    print(f"  t={tt:.2f} -> {p.name}")

cropped_images = [
    crop_white_border_soft(
        mpimg.imread(p),
        white_thr=0.992,
        pad_top=18,
        pad_bottom=10,
        pad_left=10,
        pad_right=10,
    )
    for p in img_paths
]

bg_color = "#d9d9d9"

fig = plt.figure(figsize=(18, 12), facecolor=bg_color)
gs = fig.add_gridspec(2, 6)

axes = [
    fig.add_subplot(gs[0, 0:2]),
    fig.add_subplot(gs[0, 2:4]),
    fig.add_subplot(gs[0, 4:6]),
    fig.add_subplot(gs[1, 1:3]),
    fig.add_subplot(gs[1, 3:5]),
]

for ax, img, tt in zip(axes, cropped_images, times):
    card = FancyBboxPatch(
        (-0.04, -0.04), 1.08, 1.08,
        boxstyle="round,pad=0.02",
        transform=ax.transAxes,
        facecolor=bg_color,
        edgecolor="none",
        zorder=-10,
        clip_on=False,
    )
    ax.add_patch(card)

    ax.imshow(img)
    ax.set_title(f"t = {tt:.2f}", fontsize=18, pad=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor(bg_color)

    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle("Reconstrução POD multivariável em instantes selecionados", fontsize=24, y=0.985)

plt.subplots_adjust(
    left=0.03,
    right=0.97,
    top=0.90,
    bottom=0.04,
    wspace=0.10,
    hspace=0.16,
)

fig.savefig(
    out_path,
    dpi=220,
    bbox_inches="tight",
    pad_inches=0.04,
    facecolor=fig.get_facecolor(),
)
plt.show()

print(f"\nPainel temporal salvo: {out_path}")


Figuras encontradas em ordem temporal:
  t=0.00 -> comparacao_pod_multivariavel_t_0p00.png
  t=0.25 -> comparacao_pod_multivariavel_t_0p25.png
  t=0.50 -> comparacao_pod_multivariavel_t_0p50.png
  t=0.75 -> comparacao_pod_multivariavel_t_0p75.png
  t=1.00 -> comparacao_pod_multivariavel_t_1p00.png

Painel temporal salvo: /home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000/POD/resultados/figuras_comparativas_pod_multivariavel_tempos_0a1_20260928_064424_008286/painel_temporal_reconstrucao_pod.png


/tmp/ipykernel_14518/1704947958.py:150: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
